In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import pandas as pd
import numpy as np
import math
import plotly.express as px

from cryptosense.data import (
    candles_to_dataframe,
    drop_incomplete_candles,
    fetch_ohlcv_history,
    load_processed,
    save_dataset,
    validate_ohlcv,
)

DATA_DIR = Path.cwd().parent / "data"

In [2]:
df = load_processed("BTC/USDT", DATA_DIR)

In [3]:
df["simple_return"]=df["close"].pct_change()
df["log_return"]=np.log(df["close"] / df["close"].shift(1))



Simple vs. log returns

Observation: On BTC's worst day in the data, 2020-03-12, the simple return was −0.395 (a 39.5% fall) and the log return was −0.503. Recovering from that fall required a +65.3% gain. The sum of all daily log returns equals ln(last close / first close), confirming that log returns add up over time.

Interpretation: For small daily moves the two measures are nearly identical, but they diverge on large moves, and losses look larger in log terms because they take bigger gains to recover. Log returns are the better choice for anything summed over time, such as momentum and volatility. Simple returns remain the right choice for reporting gains and losses and for combining assets in a portfolio (Phase 6).

In [4]:
df[["close", "simple_return", "log_return"]].head()
df["simple_return"].nlargest(5)



date
2017-12-07 00:00:00+00:00    0.225014
2021-02-08 00:00:00+00:00    0.195361
2019-04-02 00:00:00+00:00    0.171968
2019-10-25 00:00:00+00:00    0.167639
2020-03-13 00:00:00+00:00    0.162208
Name: simple_return, dtype: float64

In [5]:
df["log_return"].nsmallest(5)

date
2020-03-12 00:00:00+00:00   -0.502607
2018-01-16 00:00:00+00:00   -0.216880
2017-09-14 00:00:00+00:00   -0.212657
2022-06-13 00:00:00+00:00   -0.166998
2018-02-05 00:00:00+00:00   -0.164980
Name: log_return, dtype: float64

In [6]:
df["log_return"].sum()


np.float64(2.9468816886172564)

In [7]:
print(df["log_return"].mean())
print(df["log_return"].std())
print(df["log_return"].skew())
print(df["log_return"].kurt())


0.0008804546425507189
0.035271922181481105
-0.9551327342716002
16.00611439011819


In [8]:
fig = px.histogram(df, x="log_return", nbins=200,
                   title="BTC/USDT daily log returns")
fig.update_yaxes(type="log")
fig.show()

In [9]:
r = df["log_return"].dropna()
z = (r - r.mean()) / r.std()

rows = []
for k in [2, 3, 4, 5]:
    rows.append({
        "beyond_std": k,
        "actual_days": int((z.abs() > k).sum()),
        "expected_if_normal": round(len(z) * math.erfc(k / math.sqrt(2)), 2),
    })
pd.DataFrame(rows)

,beyond_std,actual_days,expected_if_normal
0,2,190,152.29
1,3,55,9.04
2,4,19,0.21
3,5,5,0.00


### Fat tails

**Observation:** BTC had 24 days with moves beyond 4 standard deviations, where a normal distribution predicts about 0.2, and 5 days beyond 5 standard deviations, where it predicts about 0.002. Excess kurtosis of daily log returns is 16.0 (0 for a normal distribution).

**Interpretation:** Extreme days are far more common than the bell curve assumes. Risk estimates based only on standard deviation will understate the chance of large losses. The kurtosis value is driven by a handful of extreme days, so it confirms fat tails rather than measuring them precisely.

In [10]:
df["vol_20"]=df["log_return"].rolling(20).std()
df["vol_60"]=df["log_return"].rolling(60).std()

In [11]:
i=100
manual = df["log_return"].iloc[i-19:i+1].std()
rolling_value = df["vol_20"].iloc[100]
print(manual, rolling_value, np.isclose(manual, rolling_value))

0.05291569502440261 0.052915695024402604 True


### Volatility clustering

**Observation:** The 20- and 60-day rolling volatility is not constant. It drifted down through most of 2018 into a quiet autumn, then jumped around nov 2018. After 2020-03-12, `vol_20` dropped sharply around 2020-04-01 and `vol_60` around 2020-05-11, exactly when the crash day left each window. In March 2020, `vol_20` reacted sooner and more strongly than `vol_60`.

**Interpretation:** Calm and turbulent periods each persist for weeks, so recent volatility is informative about near-future volatility. This makes a simple historical-volatility baseline hard to beat in Phase 4. The drop-off cliffs are an artefact of equal-weight windows, not market events. Shorter windows trade stability for responsiveness.

In [12]:
vol_annual = df[["vol_20", "vol_60"]] * np.sqrt(365) * 100
fig = px.line(vol_annual, title="BTC/USDT rolling volatility (annualised, %)")
fig.show()

In [13]:
df["sma_20"]=df["close"].rolling(20).mean()
df["sma_100"]=df["close"].rolling(100).mean()

df["dist_sma_20"]=(df["close"]/df["sma_20"])-1
df["dist_sma_100"]=(df["close"]/df["sma_100"])-1

In [14]:
fig = px.line(df[["close", "sma_20", "sma_100"]], log_y=True,
              title="BTC/USDT close with 20- and 100-day SMAs")
fig.show()

In [15]:
df[["dist_sma_20", "dist_sma_100"]].describe()

,dist_sma_20,dist_sma_100
count,3329.000000,3249.000000
mean,0.010318,0.053875
std,0.090555,0.251612
min,-0.441886,-0.473511
25%,-0.037738,-0.105443
50%,0.004335,0.018843
75%,0.053735,0.175944
max,0.662698,1.698030


### Moving-average lag

**Observation:** After the March 2020 low, the close took about two months to cross back above its 100-day SMA, by which point the price had already recovered much of its fall. `dist_sma_100` has a wider range (-0.473511 to 1.698030) than `dist_sma_20` (-0.441886 to 0.662698).

**Interpretation:** Longer averages are more stable but react late to turning points, so trend signals based on them arrive after much of a move has happened. Prices can drift further from a slow average, which explains the wider range.

In [16]:
df["vol_avg_20"]=df["volume"].shift(1).rolling(20).mean()
df["rel_volume_20"]=df["volume"]/df["vol_avg_20"]
print(df["rel_volume_20"].isna().sum())

20


In [17]:
df.nlargest(5, "rel_volume_20")[["close", "log_return", "rel_volume_20"]]

top = df.nlargest(5, "rel_volume_20").copy()
top["range"] = top["high"] / top["low"] - 1
top[["close", "log_return", "rel_volume_20", "range"]]

,close,log_return,rel_volume_20,range
date,,,,
2018-11-20 00:00:00+00:00,4558.86,-0.074207,5.853465,0.166953
2024-08-05 00:00:00+00:00,54018.81,-0.073883,5.781491,0.189910
2020-03-13 00:00:00+00:00,5578.60,0.150322,5.691580,0.574510
2026-02-05 00:00:00+00:00,62909.86,-0.151026,5.084018,0.176376
2022-07-08 00:00:00+00:00,21594.75,-0.001399,4.626694,0.063150


### Relative volume

**Observation:** `rel_volume_20` has a correlation of 0.49 with the size of the daily move (absolute log return) and −0.07 with its direction. Several of the five highest relative-volume days, including 2020-03-13, had close-to-close returns near zero.

**Interpretation:** Unusual volume signals that something significant is happening, but not which way. It suits confirming a move rather than predicting direction. High volume on days with small close-to-close returns suggests large intraday swings that daily returns miss, so the intraday range may be a useful complementary measure.

In [18]:
print("with |return|:", df["rel_volume_20"].corr(df["log_return"].abs()))
print("with return:  ", df["rel_volume_20"].corr(df["log_return"]))

with |return|: 0.4928395388871522
with return:   -0.0711869574357378


In [19]:
#Momentum
df["mom_5"]=np.log(df["close"]/df["close"].shift(5))
df["mom_20"]=np.log(df["close"]/df["close"].shift(20))

print(np.allclose(df["mom_5"], df["log_return"].rolling(5).sum(), equal_nan=True)) #additivity 
print(df["mom_5"].isna().sum())

True
5


In [20]:
#correlation matrix

features = ["log_return", "mom_5", "mom_20", "dist_sma_20", "dist_sma_100",
            "vol_20", "vol_60", "rel_volume_20"]
corr = df[features].corr()

fig = px.imshow(corr, text_auto=".2f", color_continuous_scale="RdBu_r",
                zmin=-1, zmax=1, title="BTC feature correlations")
fig.show()

### Feature overlap

**Observation:** Among the features, only `mom_20` and `dist_sma_20` have a correlation above 0.8 (0.87).

**Interpretation:** Both compare today's close with the price over the last 20 days, one against a single past close and one against the 20-day average, so they largely measure the same thing. The overlap comes from the formulas rather than from BTC specifically. Using both in K-means (Phase 3) would give this information double weight, so only one should be kept for clustering.

In [21]:
#trend strength (ER)
df["er_20"]=df["mom_20"].abs()/df["log_return"].abs().rolling(20).sum()
df["er_20"].describe()

count    3328.000000
mean        0.254599
std         0.190932
min         0.000052
25%         0.095795
50%         0.214565
75%         0.374597
max         0.901298
Name: er_20, dtype: float64

In [22]:
features = ["log_return", "mom_5", "mom_20", "dist_sma_20", "dist_sma_100",
            "vol_20", "vol_60", "rel_volume_20", "er_20"]
corr = df[features].corr()

fig = px.imshow(corr, text_auto=".2f", color_continuous_scale="RdBu_r",
                zmin=-1, zmax=1, title="BTC feature correlations")
fig.show()

### Efficiency ratio

**Observation:** `er_20` ranges between 0.1 and 0.9, with 0.254599 on average. Its correlation is 0.26 with `mom_20` and 0.05 with `vol_20`.

**Interpretation:** Trend cleanliness carries information that neither momentum nor volatility captures, so it is worth keeping as a separate feature. The modest positive correlation with `mom_20` suggests clean 20-day stretches have more often been rising than falling in this sample, which may not hold for other assets or periods.